# Step 03 — PCA & Clustering

This notebook explores the PCA dimensionality reduction and k-means clustering
results produced by `pipelines/03_cluster.py`.

Two cluster assignments are produced:
- `cluster`: best k chosen by silhouette score (capped at `k_cap`)
- `balanced_cluster`: size-constrained k-means at a fixed `balanced_k`

**Prerequisites:** Run `python pipelines/03_cluster.py` first (or `python run_pipeline.py --steps 1,2,3`).

## Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

import logging

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import plotting, DATA_DIR

setup_logging()
log = logging.getLogger(__name__)

cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

print("Config loaded")
print("RunConfig:", run_cfg)

## Load Clustering Artifacts

Three parquet files are loaded:
- `cluster_labels.parquet`: quarter → cluster assignment (both `cluster` and `balanced_cluster`)
- `pca_components.parquet`: the 5 PCA components for each quarter
- `kmeans_scores.parquet`: silhouette / CH / DB scores for each k in the sweep

In [ ]:
regimes_dir = DATA_DIR / "regimes"

labels = None
pca_df = None
scores = None

try:
    labels = pd.read_parquet(regimes_dir / "cluster_labels.parquet")
    print(f"Labels loaded: {labels.shape[0]} quarters, columns: {list(labels.columns)}")
except FileNotFoundError:
    print("ERROR: cluster_labels.parquet not found. Run pipelines/03_cluster.py first.")

try:
    pca_df = pd.read_parquet(regimes_dir / "pca_components.parquet")
    print(f"PCA components loaded: {pca_df.shape[0]} quarters × {pca_df.shape[1]} components")
except FileNotFoundError:
    print("ERROR: pca_components.parquet not found. Run pipelines/03_cluster.py first.")

try:
    scores = pd.read_parquet(regimes_dir / "kmeans_scores.parquet")
    print(f"K-sweep scores loaded: {scores.shape[0]} rows, columns: {list(scores.columns)}")
except FileNotFoundError:
    print("ERROR: kmeans_scores.parquet not found. Run pipelines/03_cluster.py first.")

## PCA Explained Variance

If explained variance was saved alongside the PCA components, show it here.

In [ ]:
if pca_df is not None:
    print("PCA component columns:", list(pca_df.columns))
    print(f"\nShape: {pca_df.shape}")
    print("\nFirst 5 rows:")
    display(pca_df.head())

    # Check for explained_variance column if it was stored
    ev_path = regimes_dir / "pca_explained_variance.parquet"
    try:
        ev = pd.read_parquet(ev_path)
        print("\nPCA explained variance:")
        display(ev)
    except FileNotFoundError:
        print("(pca_explained_variance.parquet not found — explained variance not stored separately)")

## K-Sweep Elbow Curves

Three metrics are plotted across the range of k values tested.
The vertical dashed line marks the automatically chosen best k.

In [ ]:
if scores is not None:
    print("K-sweep scores:")
    display(scores)

    # Determine best_k from max silhouette score
    if "silhouette" in scores.columns and "k" in scores.columns:
        best_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
        print(f"\nBest k (max silhouette): {best_k}")
        plotting.plot_elbow_curve(scores, best_k, run_cfg)
    else:
        print("scores DataFrame missing 'silhouette' or 'k' columns — cannot plot elbow curve.")
        print(f"Available columns: {list(scores.columns)}")

## PCA Scatter Plot

Two scatter plots: PC1 vs PC2, and PC3 vs PC4.
Points are coloured by the `balanced_cluster` assignment.
Well-separated clusters indicate a clean macro regime structure.

In [ ]:
if pca_df is not None and labels is not None:
    if "balanced_cluster" in labels.columns:
        plotting.plot_pca_scatter(pca_df, labels["balanced_cluster"], {}, run_cfg)
    elif "cluster" in labels.columns:
        print("'balanced_cluster' not found — falling back to 'cluster'")
        plotting.plot_pca_scatter(pca_df, labels["cluster"], {}, run_cfg)
    else:
        print(f"No cluster column found. Available: {list(labels.columns)}")

## Cluster Sizes

Bar chart showing how many quarters fall into each cluster.
The balanced clustering constrains sizes to be roughly equal.

In [ ]:
if labels is not None:
    if "balanced_cluster" in labels.columns:
        plotting.plot_cluster_sizes(labels["balanced_cluster"], {}, run_cfg)
    elif "cluster" in labels.columns:
        print("'balanced_cluster' not found — falling back to 'cluster'")
        plotting.plot_cluster_sizes(labels["cluster"], {}, run_cfg)
    else:
        print(f"No cluster column found. Available: {list(labels.columns)}")

## Cluster Value Counts

Exact counts for both cluster assignments.

In [ ]:
if labels is not None:
    if "cluster" in labels.columns:
        print("=== cluster (silhouette-selected k) ===")
        print(labels["cluster"].value_counts().sort_index().to_string())

    if "balanced_cluster" in labels.columns:
        print("\n=== balanced_cluster (size-constrained k) ===")
        print(labels["balanced_cluster"].value_counts().sort_index().to_string())

    print("\n=== All label columns ===")
    display(labels.head(10))